# Full-Matrix Excitatory Order vs E-E Backbones

## tl;dr

This notebook compares the excitatory-only order extracted from the dated full 85-node pathfinding runs to the existing `w_ij` and `m_ij` E-E backbone orderings. The strongest agreement in the generated summary is `full_mij_min` vs `wij_EE_min` with Spearman rho `0.689` across `28` shared excitatory nodes. The weakest agreement is `full_mij_min` vs `wij_EE_max` with Spearman rho `-0.689`.

The full-matrix order keeps inhibitory nodes in the optimization and then filters them out afterward, so the excitatory ranks reflect inhibitory modulation of the path context rather than a purely E-E optimization.


## Context & Methods

The full-matrix runs were generated with `run_full_matrix_pathfinding.py` using the `101326_` prefix over `matrices/mij_netlist.csv`. Four full-order sources are included: `w_ij` max, `w_ij` min, `m_ij` max, and `m_ij` min. E-E references are the existing max/min `w_ij` GAA backbone outputs and max/min `m_ij` backbone outputs.

Ranks are compared only on shared excitatory nodes. Full-order global positions are preserved in the detailed rank table, but rank correlations use the relative excitatory order after inhibitory nodes have been removed.


## Data

In [ ]:
from pathlib import Path
import html
import numpy as np
import pandas as pd

OUTDIR = Path("101326_excitatory_order_comparison_outputs")
rank_table = pd.read_csv(OUTDIR / "order_rank_comparison.csv")
summary = pd.read_csv(OUTDIR / "summary.csv")
top_disagreements = pd.read_csv(OUTDIR / "top_rank_disagreements.csv")

try:
    from IPython.display import HTML, SVG, display
except Exception:
    HTML = SVG = None

    def display(obj):
        print(obj)

ORDER_LABELS = {
    "full_wij_max": "full w_ij max",
    "full_wij_min": "full w_ij min",
    "full_mij_max": "full m_ij max",
    "full_mij_min": "full m_ij min",
    "wij_EE_max": "w_ij E-E max",
    "wij_EE_min": "w_ij E-E min",
    "mij_EE_max": "m_ij E-E max",
    "mij_EE_min": "m_ij E-E min",
}
FULL_ORDERS = ["full_wij_max", "full_wij_min", "full_mij_max", "full_mij_min"]
EE_ORDERS = ["wij_EE_max", "wij_EE_min", "mij_EE_max", "mij_EE_min"]

def show_svg(markup):
    if SVG is not None:
        display(SVG(markup))
    else:
        print(markup[:1000])

def _esc(value):
    return html.escape(str(value))

def _label(name, max_chars=24):
    text = ORDER_LABELS.get(name, name)
    return text if len(text) <= max_chars else text[: max_chars - 1] + "..."

def _format(value, digits=2):
    if pd.isna(value):
        return ""
    return f"{float(value):.{digits}f}"

display(summary.sort_values("spearman_rank_correlation", ascending=False))


### Source Coverage And Order Strips

In [ ]:
source_rows = []
for col in [f"{name}_rank" for name in FULL_ORDERS + EE_ORDERS]:
    source_rows.append(
        {
            "order": ORDER_LABELS[col[:-5]],
            "rank_column": col,
            "available_nodes": int(rank_table[col].notna().sum()),
        }
    )
source_coverage = pd.DataFrame(source_rows)
display(source_coverage)

def order_strips_svg(table, order_names, title):
    cell_w = 16
    row_h = 52
    left = 150
    top = 38
    max_nodes = max(int(table[f"{name}_rank"].notna().sum()) for name in order_names)
    width = left + max_nodes * cell_w + 35
    height = top + len(order_names) * row_h + 30
    parts = [
        f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'>",
        "<style>text{font-family:Segoe UI,Arial,sans-serif;font-size:11px;fill:#1f2937}.title{font-size:15px;font-weight:700}.axis{fill:#6b7280}.rank{font-size:9px;fill:#6b7280}</style>",
        f"<text class='title' x='0' y='20'>{_esc(title)}</text>",
    ]
    for row_idx, name in enumerate(order_names):
        rank_col = f"{name}_rank"
        y = top + row_idx * row_h
        order_df = table[["node", rank_col]].dropna().sort_values(rank_col)
        parts.append(f"<text x='0' y='{y + 17}'>{_esc(_label(name, 28))}</text>")
        for _, row in order_df.iterrows():
            rank = int(row[rank_col])
            x = left + (rank - 1) * cell_w
            color = "#315c9a" if "max" in name else "#b8791f"
            parts.append(
                f"<rect x='{x}' y='{y}' width='{cell_w - 2}' height='20' fill='{color}' opacity='0.88'>"
                f"<title>{_esc(row.node)} | {_esc(_label(name))} rank {rank}</title></rect>"
            )
        parts.append(f"<text class='rank' x='{left}' y='{y + 35}'>1</text>")
        parts.append(f"<text class='rank' x='{left + (len(order_df) - 1) * cell_w - 4}' y='{y + 35}'>{len(order_df)}</text>")
    parts.append("</svg>")
    return "".join(parts)

show_svg(order_strips_svg(rank_table, FULL_ORDERS, "Excitatory ranks extracted from full 85-node runs"))
show_svg(order_strips_svg(rank_table, EE_ORDERS, "Reference E-E backbone ranks"))


In [ ]:
def inhibitory_gap_svg(table, full_orders):
    rows = []
    for name in full_orders:
        global_col = f"{name}_global_position"
        rank_col = f"{name}_rank"
        temp = table[["node", global_col, rank_col]].dropna().copy()
        temp["inhibitory_slots_before_node"] = temp[global_col] - temp[rank_col]
        temp["order"] = name
        rows.append(temp)
    gaps = pd.concat(rows, ignore_index=True)
    display(
        gaps.groupby("order")
        .agg(
            max_inhibitory_slots_before_node=("inhibitory_slots_before_node", "max"),
            median_inhibitory_slots_before_node=("inhibitory_slots_before_node", "median"),
        )
        .rename(index=ORDER_LABELS)
    )

    cell_w = 20
    row_h = 55
    left = 160
    top = 42
    max_x = int(gaps["inhibitory_slots_before_node"].max())
    width = left + (max_x + 3) * cell_w + 40
    height = top + len(full_orders) * row_h + 38
    parts = [
        f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'>",
        "<style>text{font-family:Segoe UI,Arial,sans-serif;font-size:11px;fill:#1f2937}.title{font-size:15px;font-weight:700}.axis{fill:#6b7280}</style>",
        "<text class='title' x='0' y='20'>How many inhibitory nodes precede each excitatory node in the full order</text>",
    ]
    for idx, name in enumerate(full_orders):
        y = top + idx * row_h
        temp = gaps[gaps["order"].eq(name)].sort_values(f"{name}_rank")
        parts.append(f"<text x='0' y='{y + 18}'>{_esc(_label(name, 28))}</text>")
        parts.append(f"<line x1='{left}' y1='{y + 20}' x2='{left + max_x * cell_w}' y2='{y + 20}' stroke='#d1d5db'/>")
        for _, row in temp.iterrows():
            x = left + int(row["inhibitory_slots_before_node"]) * cell_w
            color = "#315c9a" if "max" in name else "#b8791f"
            parts.append(
                f"<circle cx='{x}' cy='{y + 20}' r='4' fill='{color}' opacity='0.72'>"
                f"<title>{_esc(row.node)} | {int(row['inhibitory_slots_before_node'])} inhibitory nodes before it | global position {int(row[f'{name}_global_position'])}</title></circle>"
            )
    parts.append(f"<text class='axis' x='{left}' y='{height - 10}'>0</text>")
    parts.append(f"<text class='axis' x='{left + max_x * cell_w - 16}' y='{height - 10}'>{max_x} inhibitory slots before node</text>")
    parts.append("</svg>")
    return "".join(parts)

show_svg(inhibitory_gap_svg(rank_table, FULL_ORDERS))


## Results

In [ ]:
# Broadest agreements and disagreements by rank correlation.
display(summary.sort_values("spearman_rank_correlation", ascending=False).head(8))
display(summary.sort_values("spearman_rank_correlation").head(8))


In [ ]:
def heatmap_svg(data, value_col, title, digits=2, diverging=False, reverse=False):
    rows = list(data["full_order"].drop_duplicates())
    cols = list(data["ee_backbone"].drop_duplicates())
    row_h = 48
    col_w = 112
    left = 128
    top = 58
    width = left + len(cols) * col_w + 20
    height = top + len(rows) * row_h + 40
    values = data[value_col].to_numpy(dtype=float)
    limit = max(abs(values.min()), abs(values.max())) if diverging else values.max()
    if limit == 0:
        limit = 1.0

    def color(value):
        if diverging:
            t = max(-1, min(1, value / limit))
            if t >= 0:
                r, g, b = 45, 92, 154
                alpha = 0.18 + 0.72 * t
            else:
                r, g, b = 184, 121, 31
                alpha = 0.18 + 0.72 * abs(t)
            return f"rgba({r},{g},{b},{alpha:.3f})"
        t = max(0, min(1, value / limit))
        if reverse:
            t = 1 - t
        r = int(244 - 185 * t)
        g = int(247 - 103 * t)
        b = int(251 - 58 * t)
        return f"rgb({r},{g},{b})"

    parts = [
        f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'>",
        "<style>text{font-family:Segoe UI,Arial,sans-serif;font-size:11px;fill:#1f2937}.title{font-size:15px;font-weight:700}.small{font-size:10px;fill:#6b7280}.value{font-size:12px;font-weight:700;text-anchor:middle;dominant-baseline:middle}</style>",
        f"<text class='title' x='0' y='22'>{_esc(title)}</text>",
    ]
    for c, col in enumerate(cols):
        x = left + c * col_w + col_w / 2
        parts.append(f"<text class='small' transform='translate({x - 10} {top - 8}) rotate(-35)'>{_esc(_label(col, 20))}</text>")
    for r, row in enumerate(rows):
        y = top + r * row_h
        parts.append(f"<text x='0' y='{y + 29}'>{_esc(_label(row, 28))}</text>")
        for c, col in enumerate(cols):
            match = data[(data["full_order"].eq(row)) & (data["ee_backbone"].eq(col))]
            if match.empty:
                continue
            value = float(match.iloc[0][value_col])
            x = left + c * col_w
            parts.append(f"<rect x='{x}' y='{y}' width='{col_w - 4}' height='{row_h - 4}' rx='3' fill='{color(value)}' stroke='#ffffff'/>")
            parts.append(f"<text class='value' x='{x + col_w / 2 - 2}' y='{y + row_h / 2 - 2}'>{value:.{digits}f}</text>")
    parts.append("</svg>")
    return "".join(parts)

show_svg(heatmap_svg(summary, "spearman_rank_correlation", "Spearman rank correlation", digits=2, diverging=True))
show_svg(heatmap_svg(summary, "kendall_tau_a", "Kendall tau-a", digits=2, diverging=True))
show_svg(heatmap_svg(summary, "mean_abs_rank_delta", "Mean absolute rank delta", digits=1, diverging=False, reverse=True))


In [ ]:
def correlation_bars_svg(data):
    chart = data.sort_values("spearman_rank_correlation")
    row_h = 24
    left = 225
    top = 34
    plot_w = 430
    width = left + plot_w + 95
    height = top + len(chart) * row_h + 36
    zero = left + plot_w / 2
    parts = [
        f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'>",
        "<style>text{font-family:Segoe UI,Arial,sans-serif;font-size:10px;fill:#1f2937}.title{font-size:15px;font-weight:700}.axis{fill:#6b7280}.value{font-weight:700}</style>",
        "<text class='title' x='0' y='20'>All pairwise Spearman correlations</text>",
        f"<line x1='{zero}' y1='{top - 8}' x2='{zero}' y2='{height - 28}' stroke='#111827' stroke-width='1'/>",
        f"<text class='axis' x='{left}' y='{height - 10}'>-1</text><text class='axis' x='{zero - 4}' y='{height - 10}'>0</text><text class='axis' x='{left + plot_w - 10}' y='{height - 10}'>+1</text>",
    ]
    for i, row in enumerate(chart.itertuples(index=False)):
        y = top + i * row_h
        label = f"{_label(row.full_order, 17)} vs {_label(row.ee_backbone, 17)}"
        value = float(row.spearman_rank_correlation)
        bar_x = zero if value >= 0 else zero + value * (plot_w / 2)
        bar_w = abs(value) * (plot_w / 2)
        color = "#315c9a" if value >= 0 else "#b8791f"
        parts.append(f"<text x='0' y='{y + 14}'>{_esc(label)}</text>")
        parts.append(f"<rect x='{bar_x}' y='{y + 4}' width='{bar_w}' height='14' fill='{color}' opacity='0.85'/>")
        text_x = bar_x + bar_w + 5 if value >= 0 else bar_x - 36
        parts.append(f"<text class='value' x='{text_x}' y='{y + 15}'>{value:.2f}</text>")
    parts.append("</svg>")
    return "".join(parts)

show_svg(correlation_bars_svg(summary))


In [ ]:
def rank_scatter_grid_svg(table, data):
    pairs = data.sort_values("spearman_rank_correlation", ascending=False)
    cell = 170
    margin_l = 54
    margin_t = 48
    cols = 4
    rows = int(np.ceil(len(pairs) / cols))
    width = cols * cell + 28
    height = rows * cell + 28
    parts = [
        f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'>",
        "<style>text{font-family:Segoe UI,Arial,sans-serif;font-size:9px;fill:#1f2937}.title{font-size:10px;font-weight:700}.axis{fill:#6b7280}</style>",
    ]
    for idx, row in enumerate(pairs.itertuples(index=False)):
        grid_x = (idx % cols) * cell
        grid_y = (idx // cols) * cell
        x0 = grid_x + margin_l
        y0 = grid_y + margin_t
        plot = 92
        full_col = f"{row.full_order}_rank"
        ee_col = f"{row.ee_backbone}_rank"
        pair = table[["node", full_col, ee_col]].dropna()
        max_rank = int(max(pair[full_col].max(), pair[ee_col].max()))
        parts.append(f"<text class='title' x='{grid_x + 4}' y='{grid_y + 15}'>{_esc(_label(row.full_order, 16))}</text>")
        parts.append(f"<text class='title' x='{grid_x + 4}' y='{grid_y + 28}'>vs {_esc(_label(row.ee_backbone, 16))}</text>")
        parts.append(f"<text class='axis' x='{grid_x + 4}' y='{grid_y + 42}'>rho {row.spearman_rank_correlation:.2f}</text>")
        parts.append(f"<rect x='{x0}' y='{y0}' width='{plot}' height='{plot}' fill='#f8fafc' stroke='#cbd5e1'/>")
        parts.append(f"<line x1='{x0}' y1='{y0 + plot}' x2='{x0 + plot}' y2='{y0}' stroke='#94a3b8' stroke-dasharray='3 3'/>")
        for _, point in pair.iterrows():
            x = x0 + (float(point[full_col]) - 1) / max(1, max_rank - 1) * plot
            y = y0 + plot - (float(point[ee_col]) - 1) / max(1, max_rank - 1) * plot
            parts.append(
                f"<circle cx='{x}' cy='{y}' r='2.8' fill='#315c9a' opacity='0.72'>"
                f"<title>{_esc(point.node)} | full rank {int(point[full_col])} | E-E rank {int(point[ee_col])}</title></circle>"
            )
        parts.append(f"<text class='axis' x='{x0}' y='{y0 + plot + 14}'>full rank</text>")
        parts.append(f"<text class='axis' transform='translate({x0 - 10} {y0 + plot}) rotate(-90)'>E-E rank</text>")
    parts.append("</svg>")
    return "".join(parts)

show_svg(rank_scatter_grid_svg(rank_table, summary))


### Largest Node-Level Shifts

In [ ]:
# Largest node-level shifts for the m_ij full-matrix order, where inhibitory modulation is included.
focus = top_disagreements[top_disagreements["full_order"].str.startswith("full_mij")]
display(focus.sort_values(["full_order", "ee_backbone", "abs_rank_delta"], ascending=[True, True, False]).head(32))

def disagreement_bars_svg(disagreements, full_order, ee_backbone, n=12):
    full_col = f"{full_order}_rank"
    ee_col = f"{ee_backbone}_rank"
    subset = (
        disagreements[
            disagreements["full_order"].eq(full_order)
            & disagreements["ee_backbone"].eq(ee_backbone)
        ]
        .sort_values("abs_rank_delta", ascending=True)
        .tail(n)
    )
    row_h = 25
    left = 175
    top = 38
    plot_w = 360
    max_abs = float(subset["abs_rank_delta"].max()) or 1
    zero = left + plot_w / 2
    width = left + plot_w + 80
    height = top + len(subset) * row_h + 35
    parts = [
        f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'>",
        "<style>text{font-family:Segoe UI,Arial,sans-serif;font-size:10px;fill:#1f2937}.title{font-size:15px;font-weight:700}.axis{fill:#6b7280}.value{font-weight:700}</style>",
        f"<text class='title' x='0' y='20'>Largest shifts: {_esc(_label(full_order))} vs {_esc(_label(ee_backbone))}</text>",
        f"<line x1='{zero}' y1='{top - 8}' x2='{zero}' y2='{height - 25}' stroke='#111827'/>",
    ]
    for i, row in enumerate(subset.itertuples(index=False)):
        y = top + i * row_h
        delta = float(row.rank_delta)
        bar_w = abs(delta) / max_abs * (plot_w / 2)
        bar_x = zero if delta >= 0 else zero - bar_w
        color = "#315c9a" if delta >= 0 else "#b8791f"
        node_label = row.node if len(row.node) <= 24 else row.node[:23] + "..."
        parts.append(f"<text x='0' y='{y + 15}'>{_esc(node_label)}</text>")
        parts.append(f"<rect x='{bar_x}' y='{y + 5}' width='{bar_w}' height='14' fill='{color}' opacity='0.85'><title>{_esc(row.node)} | full rank {int(getattr(row, full_col))} | E-E rank {int(getattr(row, ee_col))}</title></rect>")
        text_x = bar_x + bar_w + 5 if delta >= 0 else bar_x - 32
        parts.append(f"<text class='value' x='{text_x}' y='{y + 16}'>{delta:+.0f}</text>")
    parts.append(f"<text class='axis' x='{left}' y='{height - 8}'>earlier in full</text>")
    parts.append(f"<text class='axis' x='{zero + 70}' y='{height - 8}'>later in full</text>")
    parts.append("</svg>")
    return "".join(parts)

best_pairs = summary.sort_values("spearman_rank_correlation", ascending=False).head(4)
for row in best_pairs.itertuples(index=False):
    show_svg(disagreement_bars_svg(top_disagreements, row.full_order, row.ee_backbone, n=10))


In [ ]:
def slope_svg(table, full_order, ee_backbone, nodes=None):
    full_col = f"{full_order}_rank"
    ee_col = f"{ee_backbone}_rank"
    pair = table[["node", full_col, ee_col]].dropna().copy()
    pair["delta"] = pair[full_col] - pair[ee_col]
    if nodes is None:
        nodes = pair.reindex(pair["delta"].abs().sort_values(ascending=False).index).head(14)["node"].tolist()
    pair = pair[pair["node"].isin(nodes)].copy()
    pair = pair.sort_values(ee_col)
    max_rank = int(max(pair[full_col].max(), pair[ee_col].max()))
    row_h = 21
    left_x = 180
    right_x = 470
    top = 52
    height = top + max_rank * row_h + 28
    width = 650
    parts = [
        f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'>",
        "<style>text{font-family:Segoe UI,Arial,sans-serif;font-size:10px;fill:#1f2937}.title{font-size:15px;font-weight:700}.axis{font-size:11px;font-weight:700}</style>",
        f"<text class='title' x='0' y='20'>Selected rank movements: {_esc(_label(full_order))} vs {_esc(_label(ee_backbone))}</text>",
        f"<text class='axis' x='{left_x - 45}' y='42'>{_esc(_label(full_order))}</text>",
        f"<text class='axis' x='{right_x - 40}' y='42'>{_esc(_label(ee_backbone))}</text>",
    ]
    for _, row in pair.iterrows():
        y_full = top + (int(row[full_col]) - 1) * row_h
        y_ee = top + (int(row[ee_col]) - 1) * row_h
        color = "#315c9a" if row["delta"] >= 0 else "#b8791f"
        node_label = row.node if len(row.node) <= 30 else row.node[:29] + "..."
        parts.append(f"<line x1='{left_x}' y1='{y_full}' x2='{right_x}' y2='{y_ee}' stroke='{color}' stroke-width='1.7' opacity='0.75'/>")
        parts.append(f"<circle cx='{left_x}' cy='{y_full}' r='3' fill='{color}'/>")
        parts.append(f"<circle cx='{right_x}' cy='{y_ee}' r='3' fill='{color}'/>")
        parts.append(f"<text x='0' y='{y_full + 3}'>{_esc(node_label)}</text>")
        parts.append(f"<text x='{right_x + 12}' y='{y_ee + 3}'>{_esc(node_label)}</text>")
    parts.append("</svg>")
    return "".join(parts)

for row in summary.sort_values("spearman_rank_correlation", ascending=False).head(2).itertuples(index=False):
    show_svg(slope_svg(rank_table, row.full_order, row.ee_backbone))


## Takeaways

Use `summary.csv` for the pairwise rank-correlation view and `top_rank_disagreements.csv` to inspect the neuron types whose positions move the most when the full inhibitory context is included. Because the full-matrix min objective can exploit zero or very weak adjacent steps, interpret min-order comparisons as a stress-test ordering rather than a direct biological flow estimate.
